# 6. SVD -- Intuition and Applications

The **Singular Value Decomposition** (SVD) is arguably the most important factorisation in all of applied linear algebra.

Every matrix $A$ (any shape, any rank) can be decomposed as:

$$A = U \Sigma V^T$$

- $U$ -- orthogonal matrix (rotation/reflection in output space)
- $\Sigma$ -- diagonal matrix of **singular values** $\sigma_1 \ge \sigma_2 \ge \ldots \ge 0$
- $V^T$ -- orthogonal matrix (rotation/reflection in input space)

Geometrically: **rotate, scale along axes, rotate again**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline

BLUE  = "#2196F3"
RED   = "#F44336"
GREEN = "#4CAF50"
ORANGE = "#FF9800"
GRAY  = "#BDBDBD"

def setup_ax(ax, lim=3, title=""):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.grid(True, lw=0.4, alpha=0.5)
    ax.axhline(0, color="k", lw=0.4)
    ax.axvline(0, color="k", lw=0.4)
    if title:
        ax.set_title(title, fontsize=12)

## 6.1 SVD Step-by-Step on the Unit Circle

We visualise how $A = U \Sigma V^T$ transforms the unit circle through three stages:

1. $V^T$: rotate the circle (stays a circle)
2. $\Sigma$: stretch into an axis-aligned ellipse
3. $U$: rotate/reflect the ellipse to its final orientation

In [ ]:
def make_unit_circle(n=200):
    theta = np.linspace(0, 2 * np.pi, n)
    return np.column_stack([np.cos(theta), np.sin(theta)])


def plot_svd_steps(A):
    """
    Show the 4-panel SVD decomposition: original, after V^T, after Sigma, after U.

    Parameters
    ----------
    A : ndarray (2, 2)
    """
    U, s, Vt = np.linalg.svd(A)
    S = np.diag(s)

    circle = make_unit_circle()
    basis = np.eye(2)

    stages = [
        (circle, basis, "1. Unit circle"),
        (circle @ Vt.T, basis @ Vt.T, "2. After V^T (rotate)"),
        (
            circle @ Vt.T @ S.T,
            basis @ Vt.T @ S.T,
            f"3. After Σ (scale by {s[0]:.2f}, {s[1]:.2f})",
        ),
        (
            circle @ Vt.T @ S.T @ U.T,
            basis @ Vt.T @ S.T @ U.T,
            "4. After U (rotate) = A·circle",
        ),
    ]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    lim = max(s[0] + 1, 3)

    for ax, (pts, bas, title) in zip(axes, stages):
        setup_ax(ax, lim=lim, title=title)
        ax.fill(pts[:, 0], pts[:, 1], alpha=0.2, color=BLUE)
        ax.plot(pts[:, 0], pts[:, 1], color=BLUE, lw=1.5)
        ax.annotate(
            "",
            xy=bas[0],
            xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color=RED, lw=2),
        )
        ax.annotate(
            "",
            xy=bas[1],
            xytext=(0, 0),
            arrowprops=dict(arrowstyle="->", color=GREEN, lw=2),
        )

    plt.tight_layout()
    plt.show()

    print(f"\nSVD of A:")
    print(f"  U     = \n{np.round(U, 4)}")
    print(f"  Σ     = diag({np.round(s, 4)})")
    print(f"  V^T   = \n{np.round(Vt, 4)}")
    print(f"  U Σ V^T = \n{np.round(U @ S @ Vt, 4)}")
    print(f"  A       = \n{np.round(A, 4)}")

In [ ]:
A = np.array([[2, 1], [1, 2]])
plot_svd_steps(A)

In [ ]:
A2 = np.array([[1.5, 0.8], [0.3, 2.0]])
plot_svd_steps(A2)

### Interactive SVD explorer

In [ ]:
def interactive_svd(a, b, c, d):
    A = np.array([[a, b], [c, d]])
    plot_svd_steps(A)


style = {"description_width": "initial"}
widgets.interact(
    interactive_svd,
    a=widgets.FloatSlider(
        value=2, min=-3, max=3, step=0.1, description="a", style=style
    ),
    b=widgets.FloatSlider(
        value=1, min=-3, max=3, step=0.1, description="b", style=style
    ),
    c=widgets.FloatSlider(
        value=1, min=-3, max=3, step=0.1, description="c", style=style
    ),
    d=widgets.FloatSlider(
        value=2, min=-3, max=3, step=0.1, description="d", style=style
    ),
)

## 6.2 Singular Values, Eigenvalues, and the Connection

For any matrix $A$:
- The **singular values** of $A$ are the square roots of the eigenvalues of $A^T A$ (or $A A^T$).
- The columns of $V$ are the eigenvectors of $A^T A$.
- The columns of $U$ are the eigenvectors of $A A^T$.

Unlike eigendecomposition, SVD works for **any** matrix (non-square, singular, etc.).

In [ ]:
A = np.array([[2, 1], [1, 2]])

U, s, Vt = np.linalg.svd(A)

AtA = A.T @ A
eig_vals_AtA, eig_vecs_AtA = np.linalg.eigh(AtA)

print("Singular values of A:", s)
print("Eigenvalues of A^T A:", eig_vals_AtA)
print("sqrt(eigenvalues):   ", np.sqrt(eig_vals_AtA[::-1]))
print("\nMatch:", np.allclose(s, np.sqrt(eig_vals_AtA[::-1])))

## 6.3 Low-Rank Approximation (Image Compression)

One of the most powerful applications of SVD is the **low-rank approximation**:

$$A \approx \sum_{i=1}^{r} \sigma_i \, \mathbf{u}_i \, \mathbf{v}_i^T$$

Using only the top $r$ singular values gives the best rank-$r$ approximation (in Frobenius or spectral norm).  
We demonstrate this by compressing a synthetically generated grayscale image.

In [ ]:
def make_sample_image(size=128):
    """Generate a synthetic grayscale image with geometric patterns."""
    x = np.linspace(-3, 3, size)
    X, Y = np.meshgrid(x, x)
    img = (
        np.sin(2 * X) * np.cos(2 * Y)
        + 0.5 * np.exp(-((X - 1) ** 2 + (Y - 1) ** 2))
        - 0.5 * np.exp(-((X + 1) ** 2 + (Y + 1) ** 2))
        + 0.3 * np.sin(5 * X + 3 * Y)
    )
    img = (img - img.min()) / (img.max() - img.min())
    return img


def svd_compress(img, rank):
    """Compress image using rank-r SVD approximation."""
    U, s, Vt = np.linalg.svd(img, full_matrices=False)
    return U[:, :rank] @ np.diag(s[:rank]) @ Vt[:rank, :]


img = make_sample_image()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
ranks = [1, 3, 10, 30, img.shape[0]]
U, s, Vt = np.linalg.svd(img, full_matrices=False)

for ax, r in zip(axes, ranks):
    compressed = svd_compress(img, r)
    err = np.linalg.norm(img - compressed, "fro") / np.linalg.norm(img, "fro")
    ax.imshow(compressed, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"rank {r}\nerror: {err:.3f}", fontsize=11)
    ax.axis("off")

plt.suptitle("SVD Image Compression at different ranks", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(s, "o-", markersize=3, color=BLUE)
ax.set_xlabel("Singular value index")
ax.set_ylabel("σ (log scale)")
ax.set_title("Singular value spectrum")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

energy = np.cumsum(s**2) / np.sum(s**2)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(energy, "o-", markersize=3, color=RED)
ax.axhline(0.95, ls="--", color="gray", alpha=0.6, label="95% energy")
rank_95 = np.searchsorted(energy, 0.95) + 1
ax.axvline(rank_95, ls="--", color="gray", alpha=0.6)
ax.set_xlabel("Number of singular values kept")
ax.set_ylabel("Fraction of total energy")
ax.set_title(f"Cumulative energy -- 95% captured at rank {rank_95}")
ax.grid(True, alpha=0.4)
ax.legend()
plt.tight_layout()
plt.show()

### Interactive rank slider

In [ ]:
def interactive_compress(rank):
    compressed = svd_compress(img, rank)
    err = np.linalg.norm(img - compressed, "fro") / np.linalg.norm(img, "fro")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
    ax1.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax1.set_title("Original")
    ax1.axis("off")
    ax2.imshow(compressed, cmap="gray", vmin=0, vmax=1)
    ax2.set_title(f"Rank-{rank} approximation (error: {err:.4f})")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()


widgets.interact(
    interactive_compress,
    rank=widgets.IntSlider(
        value=5, min=1, max=img.shape[0], step=1, description="Rank"
    ),
)

## 6.4 Pseudo-Inverse and Least Squares

The **Moore-Penrose pseudo-inverse** $A^+$ generalises matrix inversion to non-square and singular matrices:

$$A^+ = V \Sigma^+ U^T$$

where $\Sigma^+$ inverts only the non-zero singular values.

For an overdetermined system $A\mathbf{x} = \mathbf{b}$ (more equations than unknowns), $\mathbf{x} = A^+ \mathbf{b}$ gives the **least-squares** solution.

### Line Fitting Example

Given noisy data points, we fit a line $y = mx + c$ by setting up the overdetermined system and solving via pseudo-inverse.

In [ ]:
rng = np.random.default_rng(42)
n_pts = 30
x_data = np.linspace(-3, 3, n_pts)
y_true = 1.5 * x_data + 0.5
y_data = y_true + rng.normal(scale=0.8, size=n_pts)

A_ls = np.column_stack([x_data, np.ones(n_pts)])

# Solve via SVD pseudo-inverse
params_pinv = np.linalg.pinv(A_ls) @ y_data

# Compare with numpy's lstsq
params_lstsq, _, _, _ = np.linalg.lstsq(A_ls, y_data, rcond=None)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_data, y_data, color=BLUE, s=30, label="Noisy data")
ax.plot(x_data, y_true, "--", color=GRAY, lw=1.5, label="True line")
ax.plot(
    x_data,
    A_ls @ params_pinv,
    color=RED,
    lw=2,
    label=f"SVD fit: y = {params_pinv[0]:.2f}x + {params_pinv[1]:.2f}",
)
ax.legend(fontsize=11)
ax.set_title("Least-squares line fitting via pseudo-inverse")
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(
    f"Pseudo-inverse solution: m={params_pinv[0]:.4f}, c={params_pinv[1]:.4f}"
)
print(
    f"lstsq solution:         m={params_lstsq[0]:.4f}, c={params_lstsq[1]:.4f}"
)

## 6.5 PCA -- Principal Component Analysis

PCA finds the directions of maximum variance in data.  It is directly computed from the SVD of the **centred** data matrix.

Given data matrix $X$ (rows = samples, columns = features), centred so each column has zero mean:

$$X = U \Sigma V^T$$

The columns of $V$ are the **principal components** (directions of maximum variance).  
The singular values (squared and divided by $n-1$) give the variance along each component.

In [ ]:
rng = np.random.default_rng(7)
n_samples = 200

cov = np.array([[3, 1.5], [1.5, 1]])
data = rng.multivariate_normal([0, 0], cov, size=n_samples)

mean = data.mean(axis=0)
X_centred = data - mean
U, s, Vt = np.linalg.svd(X_centred, full_matrices=False)

fig, ax = plt.subplots(figsize=(7, 7))
setup_ax(ax, lim=6, title="PCA: principal components from SVD")
ax.scatter(data[:, 0], data[:, 1], s=10, alpha=0.5, color=BLUE)

colors_pc = [RED, GREEN]
for i in range(2):
    pc = Vt[i]
    variance = s[i] ** 2 / (n_samples - 1)
    length = 2 * np.sqrt(variance)
    ax.annotate(
        "",
        xy=mean + length * pc,
        xytext=mean,
        arrowprops=dict(arrowstyle="->", color=colors_pc[i], lw=3),
    )
    ax.text(
        *(mean + length * pc + 0.2),
        f"PC{i+1} (var={variance:.2f})",
        color=colors_pc[i],
        fontsize=11,
        fontweight="bold",
    )

ax.plot(*mean, "ko", ms=6)
plt.tight_layout()
plt.show()

print(f"Singular values: {s}")
print(f"Variance explained: {s**2 / (n_samples-1)}")
print(f"Total variance: {np.sum(s**2 / (n_samples-1)):.4f}")

## 6.6 Non-Square Matrices

SVD works for any $m \times n$ matrix.  For a "tall" matrix ($m > n$), the transformation maps from $\mathbb{R}^n$ into a subspace of $\mathbb{R}^m$.  For a "wide" matrix ($m < n$), information is lost.

In [ ]:
A_tall = np.array([[1, 2], [3, 4], [5, 6]])

U, s, Vt = np.linalg.svd(A_tall, full_matrices=False)
print(f"A (3×2):\n{A_tall}\n")
print(f"U (3×2):\n{np.round(U, 4)}")
print(f"Σ = diag({np.round(s, 4)})")
print(f"V^T (2×2):\n{np.round(Vt, 4)}")
print(f"\nReconstruction U Σ V^T:\n{np.round(U @ np.diag(s) @ Vt, 4)}")

---

## Exercises

### Exercise 1

Compute the SVD of $A = \begin{pmatrix} 3 & 0 \\ 0 & 2 \end{pmatrix}$ by hand.  
What are $U$, $\Sigma$, $V^T$?  Verify with numpy.

<details>
<summary><b>Solution</b></summary>

$A$ is already diagonal with positive entries, so $U = I$, $\Sigma = A$, $V^T = I$.  
The singular values are $\sigma_1 = 3, \sigma_2 = 2$.

</details>

In [ ]:
A = np.array([[3, 0], [0, 2]])
U, s, Vt = np.linalg.svd(A)
print(f"U:\n{U}")
print(f"Σ: {s}")
print(f"V^T:\n{Vt}")

### Exercise 2

Create a rank-1 matrix $B = \sigma_1 \mathbf{u}_1 \mathbf{v}_1^T$ from the leading singular triplet of

$$A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$$

How close is $B$ to $A$?  Compute the relative Frobenius-norm error.

<details>
<summary><b>Solution</b></summary>

The relative error is $\|A - B\|_F / \|A\|_F = \sigma_2 / \sqrt{\sigma_1^2 + \sigma_2^2}$.

</details>

In [ ]:
A = np.array([[1, 2], [3, 4]], dtype=float)
U, s, Vt = np.linalg.svd(A)
B = s[0] * np.outer(U[:, 0], Vt[0])
err = np.linalg.norm(A - B, "fro") / np.linalg.norm(A, "fro")
print(f"Rank-1 approximation B:\n{np.round(B, 4)}")
print(f"Relative error: {err:.6f}")
print(f"σ2/sqrt(σ1²+σ2²) = {s[1] / np.sqrt(np.sum(s**2)):.6f}")

### Exercise 3

Generate 50 noisy data points around the line $y = -0.5x + 2$ with noise $\sigma = 0.5$.  
Fit the line using the pseudo-inverse.  Compare the fitted parameters with the true values.

<details>
<summary><b>Solution</b></summary>

Set up $A = [x, 1]$ and solve $A [m, c]^T = y$ via `np.linalg.pinv(A) @ y`.  
The fitted $m$ should be close to $-0.5$ and $c$ close to $2$.

</details>

In [ ]:
rng = np.random.default_rng(99)
x = np.linspace(-3, 3, 50)
y = -0.5 * x + 2 + rng.normal(scale=0.5, size=50)

A = np.column_stack([x, np.ones_like(x)])
params = np.linalg.pinv(A) @ y
print(f"Fitted: m = {params[0]:.4f}, c = {params[1]:.4f}")
print(f"True:   m = -0.5000, c = 2.0000")